<a href="https://colab.research.google.com/github/tmedeirosb/tsi-am/blob/main/MODULO_02_plot_column_transformer_mixed_types_pt_br.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformador de Colunas com Tipos Mistos

.. currentmodule:: sklearn

Este exemplo ilustra como aplicar diferentes pipelines de pré-processamento e extração de características para diferentes subconjuntos de recursos, usando o `:class:~compose.ColumnTransformer`. Isso é particularmente útil no caso de conjuntos de dados que contêm tipos de dados heterogêneos, pois podemos querer dimensionar os recursos numéricos e codificar em *one-hot* os categóricos.

Neste exemplo, os dados numéricos são padronizados (*standard-scaled*) após a imputação da média. Os dados categóricos são codificados em *one-hot* via ``OneHotEncoder``, que cria uma nova categoria para valores ausentes. Reduzimos ainda mais a dimensionalidade selecionando categorias usando um teste qui-quadrado.

Além disso, mostramos duas maneiras diferentes de despachar as colunas para o pré-processador específico: por nomes de colunas e por tipos de dados de colunas.

Finalmente, o pipeline de pré-processamento é integrado em um pipeline de predição completo usando o `:class:~pipeline.Pipeline`, juntamente com um modelo de classificação simples.

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(0)

Carregar dados de https://www.openml.org/d/40945

In [ ]:
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)

# Alternatively X and y can be obtained directly from the frame attribute:
# X = titanic.frame.drop('survived', axis=1)
# y = titanic.frame['survived']

Use o ``ColumnTransformer`` selecionando colunas por nomes

Treinaremos nosso classificador com os seguintes recursos:

Recursos Numéricos:

* ``age``: float;
* ``fare``: float.

Recursos Categóricos:

* ``embarked``: categorias codificadas como strings ``{'C', 'S', 'Q'}``;
* ``sex``: categorias codificadas como strings ``{'female', 'male'}``;
* ``pclass``: inteiros ordinais ``{1, 2, 3}``.

Criamos os pipelines de pré-processamento para dados numéricos e categóricos.
Note que ``pclass`` pode ser tratado como um recurso categórico ou numérico.

In [ ]:
numeric_features = ["age", "fare"]
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

categorical_features = ["embarked", "sex", "pclass"]
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ("selector", SelectPercentile(chi2, percentile=50)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

Anexar o classificador ao pipeline de pré-processamento.
Agora temos um pipeline de predição completo.

In [ ]:
clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

clf.fit(X_train, y_train)
print("model score: %.3f" % clf.score(X_test, y_test))

Representação HTML do ``Pipeline`` (exibição de diagrama)

Quando o ``Pipeline`` é impresso em um notebook jupyter, uma representação HTML do estimador é exibida:

In [ ]:
clf

Use o ``ColumnTransformer`` selecionando colunas por tipos de dados

Ao lidar com um conjunto de dados limpo, o pré-processamento pode ser automático usando os tipos de dados da coluna para decidir se deve tratar uma coluna como numérica ou categórica.
O `:func:sklearn.compose.make_column_selector` oferece essa possibilidade.
Primeiro, vamos selecionar apenas um subconjunto de colunas para simplificar nosso exemplo.

In [ ]:
subset_feature = ["embarked", "sex", "pclass", "age", "fare"]
X_train, X_test = X_train[subset_feature], X_test[subset_feature]

Em seguida, fazemos a introspecção das informações sobre o tipo de dados de cada coluna.

In [ ]:
X_train.info()

Podemos observar que as colunas `embarked` e `sex` foram marcadas como colunas de `category` ao carregar os dados com o ``fetch_openml``. Portanto, podemos usar essa informação para despachar as colunas categóricas para o ``categorical_transformer`` e as colunas restantes para o ``numerical_transformer``.

<div class="alert alert-info"><h4>Nota</h4><p>Na prática, você mesmo terá que lidar com o tipo de dados da coluna.
   Se você quiser que algumas colunas sejam consideradas como `category`, terá que convertê-las em colunas categóricas. Se estiver usando pandas, pode consultar a documentação sobre [Dados Categóricos](https://pandas.pydata.org/pandas-docs/stable/user_guide/categorical.html).</p></div>

In [ ]:
from sklearn.compose import make_column_selector as selector

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, selector(dtype_exclude="category")),
        ("cat", categorical_transformer, selector(dtype_include="category")),
    ]
)
clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())]
)


clf.fit(X_train, y_train)
print("model score: %.3f" % clf.score(X_test, y_test))
clf

O score resultante não é exatamente o mesmo que o do pipeline anterior porque o seletor baseado em dtype trata a coluna ``pclass`` como um recurso numérico em vez de um recurso categórico como anteriormente:

In [ ]:
selector(dtype_exclude="category")(X_train)

In [ ]:
selector(dtype_include="category")(X_train)

Usando o pipeline de predição em uma busca em grade (Grid Search)

A busca em grade também pode ser realizada nos diferentes passos de pré-processamento definidos no objeto ``ColumnTransformer``, juntamente com os hiperparâmetros do classificador como parte do ``Pipeline``.
Buscaremos tanto a estratégia do imputer do pré-processamento numérico quanto o parâmetro de regularização da regressão logística usando o `:class:~sklearn.model_selection.RandomizedSearchCV`. Esta busca de hiperparâmetros seleciona aleatoriamente um número fixo de configurações de parâmetros configurados por `n_iter`. Alternativamente, pode-se usar o `:class:~sklearn.model_selection.GridSearchCV`, mas o produto cartesiano do espaço de parâmetros será avaliado.

In [ ]:
param_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "preprocessor__cat__selector__percentile": [10, 30, 50, 70],
    "classifier__C": [0.1, 1.0, 10, 100],
}

search_cv = RandomizedSearchCV(clf, param_grid, n_iter=10, random_state=0)
search_cv

Chamar 'fit' aciona a busca validada cruzada pela melhor combinação de hiperparâmetros:

In [ ]:
search_cv.fit(X_train, y_train)

print("Best params:")
print(search_cv.best_params_)

As pontuações de validação cruzada interna obtidas por esses parâmetros são:

In [ ]:
print(f"Internal CV score: {search_cv.best_score_:.3f}")

Também podemos inspecionar os principais resultados da busca em grade como um dataframe do pandas:

In [ ]:
import pandas as pd

cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)
cv_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_preprocessor__num__imputer__strategy",
        "param_preprocessor__cat__selector__percentile",
        "param_classifier__C",
    ]
].head(5)

Os melhores hiperparâmetros foram usados para ajustar novamente um modelo final em todo o conjunto de treinamento. Podemos avaliar esse modelo final em dados de teste reservados que não foram usados para o ajuste de hiperparâmetros.

In [ ]:
print(
    "accuracy of the best model from randomized search: "
    f"{search_cv.score(X_test, y_test):.3f}"
)